# 🚀 Full BirdBench Pipeline Runner (Train + Dev)

Use this notebook to **fine-tune on the full BirdBench dataset** and run the interactive demo.

### ⚠️ Prerequisites
Ensure you have uploaded the following to your Google Drive (`MyDrive` root):
1. **`text-to-sql-code.zip`**: Your project code (without data).
2. **`train.zip`**: The full training dataset (~8GB).
3. **`dev.zip`**: The dev dataset (~500MB).

### ⚙️ Setup Instructions
1. Go to **Runtime > Change runtime type** -> Select **T4 GPU**.

In [ ]:
# 1. Setup Environment & Code
from google.colab import drive
import os
import shutil

# Mount Drive
drive.mount('/content/drive')

# Copy and Unzip Code
ZIP_NAME = 'text-to-sql-code.zip'
DRIVE_PATH = f'/content/drive/MyDrive/{ZIP_NAME}'

if os.path.exists(DRIVE_PATH):
    print("Found code zip in Drive. Copying...")
    shutil.copy(DRIVE_PATH, ZIP_NAME)
    print("Unzipping...")
    !unzip -o -q {ZIP_NAME}
    
    # Enter directory (logic to find the folder if it extracted into a subfolder)
    if os.path.exists('text-to-SQL'):
        %cd text-to-SQL
    elif os.path.exists('text-to-sql'):
        %cd text-to-sql
        
    print("Installing dependencies...")
    !pip install -r requirements.txt
    
else:
    print(f"\u274c Error: {ZIP_NAME} not found in Google Drive!")

In [ ]:
# 2. Data Setup (Train + Dev)
# This copies the large zip files from Drive and extracts them using our helper script.

!mkdir -p data

print("Copying large datasets from Drive... (This may take 1-2 minutes)")
found_data = False

if os.path.exists('/content/drive/MyDrive/train.zip'):
    !cp /content/drive/MyDrive/train.zip data/
    print(" - train.zip copied.")
    found_data = True
else:
    print(" ⚠️ train.zip not found in Drive.")

if os.path.exists('/content/drive/MyDrive/dev.zip'):
    !cp /content/drive/MyDrive/dev.zip data/
    print(" - dev.zip copied.")
else:
    print(" ⚠️ dev.zip not found in Drive.")

if found_data:
    print("Extracting datasets... (This may take 5+ minutes)")
    !python scripts/setup_full_dataset.py
else:
    print("Skipping extraction (no data found).")

In [ ]:
# 3. Run Fine-Tuning
# The script will automatically filter to the requested index range (4250-5668)

print("Starting Fine-Tuning...")
!python src/train_bird.py

In [ ]:
# 4. Interactive Pipeline Demo
# Mode 1: Text -> SQL
# Mode 2: SQL -> NoSQL

print("Starting Interactive Demo... (Enter 'q' to quit)")
!python src/full_pipeline.py

In [ ]:
# 5. (Optional) Zip and Save Model to Drive
# Run this after training to save your fine-tuned weights

import shutil
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_name = f"bird_model_checkpoint_{timestamp}"

print(f"Zipping checkpoint to {save_name}.zip...")
shutil.make_archive(save_name, 'zip', 'results/bird_finetune')

print(f"Copying to Drive...")
shutil.copy(f"{save_name}.zip", "/content/drive/MyDrive/")
print("Done!")